## load Experimental data

### Load the experimental variant table

This cell loads the raw NylC variant data from the project directory and prints the table dimensions and column names. The imported table provides variant identifiers, mutation annotations, replicate activity measurements, melting temperature measurements, and structure file references used in the downstream ANOVA-GP workflow.


In [3]:
from pathlib import Path
import pandas as pd

project_root = Path("/home/dwp46550/ba_nylon")

df = pd.read_csv(
    project_root / "matrices" / "NylC_Puetz_raw_data.CSV",
    sep=";",
    decimal=","
)
print(df.shape)

print(df.keys())

(36, 11)
Index(['variant_id', 'variant_class', 'mutations', 'n_mutations',
       'activity_pa6_1', 'activity_pa6_2', 'activity_pa6_3', 'activity_pa6_4',
       'tm_celsius_1', 'tm_celsius_2', 'strucuture_file'],
      dtype='object')


### Aggregate PA6 activity replicates

This cell converts all PA6 activity replicate columns to numeric values and aggregates them at the variant level. For each variant, it computes the number of available activity measurements, the mean PA6 activity, the sample standard deviation, and the standard error of the mean. The mean activity is used as the regression target, while the standard error is later used as variant-specific observation noise in the Gaussian process.


In [4]:
import numpy as np
import pandas as pd
#calculates replicate mean 
activity_rep_cols = [
    col for col in df.columns
    if col.startswith("activity_pa6_")
]


df[activity_rep_cols] = df[activity_rep_cols].apply(
    pd.to_numeric,
    errors="coerce",
)


df["activity_n"] = df[activity_rep_cols].count(axis=1)

df["activity_pa6"] = df[activity_rep_cols].mean(
    axis=1,
    skipna=True,
)

df["activity_sd"] = df[activity_rep_cols].std(
    axis=1,
    skipna=True,
    ddof=1,
)

df["activity_sem"] = (
    df["activity_sd"]
    / np.sqrt(df["activity_n"])
)

print(df[
    ["variant_id", "activity_n", "activity_pa6",
     "activity_sd", "activity_sem"]
].head())

  variant_id  activity_n  activity_pa6  activity_sd  activity_sem
0         WT           3     77.666667     2.309401      1.333333
1       D99G           3    144.000000    19.313208     11.150486
2       D99V           3    147.666667     6.027714      3.480102
3       D99R           2    194.000000    14.142136     10.000000
4      F134W           2    217.000000     7.071068      5.000000


## Generate position specific Kernel

### Define amino-acid descriptors and pocket representations

This cell defines the physicochemical descriptor table for the 20 canonical amino acids and standardizes each descriptor across the amino-acid alphabet. It then defines the four modeled NylC pocket positions, parses mutation strings into fixed pocket dictionaries, converts each variant into an ordered amino-acid tuple, and builds descriptor matrices for the ANOVA-GP kernel construction. The resulting representation keeps amino-acid identities at positions 99, 134, 304, and 330 instead of using simple mutation-delta features.


In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


AA_PROPERTIES_RAW = {
    "A": {"charge": 0,  "hydrophobicity": 1.8,  "volume": 88.6,  "polarity": 8.1,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.357},
    "R": {"charge": 1,  "hydrophobicity": -4.5, "volume": 173.4, "polarity": 10.5, "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.529},
    "N": {"charge": 0,  "hydrophobicity": -3.5, "volume": 114.1, "polarity": 11.6, "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.463},
    "D": {"charge": -1, "hydrophobicity": -3.5, "volume": 111.1, "polarity": 13.0, "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 1, "flexibility": 0.511},
    "C": {"charge": 0,  "hydrophobicity": 2.5,  "volume": 108.5, "polarity": 5.5,  "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 0, "flexibility": 0.346},
    "Q": {"charge": 0,  "hydrophobicity": -3.5, "volume": 143.8, "polarity": 10.5, "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.493},
    "E": {"charge": -1, "hydrophobicity": -3.5, "volume": 138.4, "polarity": 12.3, "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 1, "flexibility": 0.497},
    "G": {"charge": 0,  "hydrophobicity": -0.4, "volume": 60.1,  "polarity": 9.0,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.544},
    "H": {"charge": 0.1,"hydrophobicity": -3.2, "volume": 153.2, "polarity": 10.4, "aromatic": 1, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.323},
    "I": {"charge": 0,  "hydrophobicity": 4.5,  "volume": 166.7, "polarity": 5.2,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.462},
    "L": {"charge": 0,  "hydrophobicity": 3.8,  "volume": 166.7, "polarity": 4.9,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.365},
    "K": {"charge": 1,  "hydrophobicity": -3.9, "volume": 168.6, "polarity": 11.3, "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 0, "flexibility": 0.466},
    "M": {"charge": 0,  "hydrophobicity": 1.9,  "volume": 162.9, "polarity": 5.7,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.295},
    "F": {"charge": 0,  "hydrophobicity": 2.8,  "volume": 189.9, "polarity": 5.2,  "aromatic": 1, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.314},
    "P": {"charge": 0,  "hydrophobicity": -1.6, "volume": 112.7, "polarity": 8.0,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.509},
    "S": {"charge": 0,  "hydrophobicity": -0.8, "volume": 89.0,  "polarity": 9.2,  "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.507},
    "T": {"charge": 0,  "hydrophobicity": -0.7, "volume": 116.1, "polarity": 8.6,  "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.444},
    "W": {"charge": 0,  "hydrophobicity": -0.9, "volume": 227.8, "polarity": 5.4,  "aromatic": 1, "hbond_donor": 1, "hbond_acceptor": 0, "flexibility": 0.305},
    "Y": {"charge": 0,  "hydrophobicity": -1.3, "volume": 193.6, "polarity": 6.2,  "aromatic": 1, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.420},
    "V": {"charge": 0,  "hydrophobicity": 4.2,  "volume": 140.0, "polarity": 5.9,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.386},
}


PROPERTY_COLUMNS = [
    "charge",
    "hydrophobicity",
    "volume",
    "polarity",
    "aromatic",
    "hbond_donor",
    "hbond_acceptor",
    "flexibility",
]


aa_property_df_raw = pd.DataFrame.from_dict(
    AA_PROPERTIES_RAW,
    orient="index",
)

aa_property_df_raw = aa_property_df_raw.loc[
    sorted(aa_property_df_raw.index),
    PROPERTY_COLUMNS,
]

scaler = StandardScaler()

aa_property_df_scaled = pd.DataFrame(
    scaler.fit_transform(aa_property_df_raw),
    index=aa_property_df_raw.index,
    columns=aa_property_df_raw.columns,
)

AA_PROPERTIES_SCALED = aa_property_df_scaled.to_dict(orient="index")


WT_POCKET = {
    99: "D",
    134: "F",
    304: "D",
    330: "R",
}

POSITIONS = [99, 134, 304, 330]


def aa_descriptor_vector(aa):
    """
    Returns the standardized physicochemical descriptor vector of one amino acid.
    """
    aa = str(aa).upper()

    if aa not in AA_PROPERTIES_SCALED:
        raise ValueError(f"Unknown amino acid: {aa}")

    return np.array(
        [AA_PROPERTIES_SCALED[aa][col] for col in PROPERTY_COLUMNS],
        dtype=float,
    )


def pocket_to_aa_tuple(mutant_pocket, positions=POSITIONS, wt_pocket=WT_POCKET):
    """
    Converts a pocket dictionary into fixed positional amino-acid order.
    Example:
    {99: "R", 134: "W", 304: "M", 330: "A"}
    -> ("R", "W", "M", "A")
    """
    aa_tuple = []

    for position in positions:
        aa = mutant_pocket.get(position, wt_pocket[position])
        aa_tuple.append(str(aa).upper())

    return tuple(aa_tuple)


def pocket_to_descriptor_matrix(mutant_pocket, positions=POSITIONS):
    """
    Returns a descriptor matrix for one variant.

    Rows: positions
    Columns: physicochemical properties

    Shape:
    (n_positions, n_properties)
    """
    aa_tuple = pocket_to_aa_tuple(
        mutant_pocket,
        positions=positions,
    )

    return np.vstack([
        aa_descriptor_vector(aa)
        for aa in aa_tuple
    ])


def parse_mutation_string_to_pocket(mutation_string, wt_pocket=WT_POCKET):
    """
    Parses mutation strings like:
    'D99R/F134W/D304M/R330A'
    'D99R, F134W, D304M, R330A'
    'F134W D304M R330A'

    Returns:
    {99: ..., 134: ..., 304: ..., 330: ...}
    """
    import re

    mutant_pocket = dict(wt_pocket)

    if pd.isna(mutation_string) or str(mutation_string).strip().lower() in [
        "",
        "wt",
        "wildtype",
        "wild type",
    ]:
        return mutant_pocket

    pattern = r"([A-Z])(\d+)([A-Z])"
    matches = re.findall(pattern, str(mutation_string).upper())

    for wt_aa, pos, mut_aa in matches:
        pos = int(pos)

        if pos not in mutant_pocket:
            continue

        expected_wt = wt_pocket[pos]

        if wt_aa != expected_wt:
            print(
                f"Warning: mutation {wt_aa}{pos}{mut_aa} "
                f"does not match expected WT {expected_wt}{pos}"
            )

        mutant_pocket[pos] = mut_aa

    return mutant_pocket


def pockets_to_position_descriptor_arrays(pockets, positions=POSITIONS):
    """
    Converts a list of pocket dictionaries into position-specific descriptor arrays.

    Output:
    position_arrays[position].shape = (n_variants, n_properties)
    """
    position_arrays = {}

    for position in positions:
        vectors = []

        for pocket in pockets:
            aa = pocket.get(position, WT_POCKET[position])
            vectors.append(aa_descriptor_vector(aa))

        position_arrays[position] = np.vstack(vectors)

    return position_arrays


def add_pocket_representations_from_mutation_column(
    df,
    mutation_col="mutations",
):
    """
    Adds ANOVA-GP compatible pocket representations.
    Does not create delta features.
    """
    df_out = df.copy()

    df_out["pocket"] = df_out[mutation_col].apply(
        parse_mutation_string_to_pocket
    )

    df_out["aa_tuple"] = df_out["pocket"].apply(
        pocket_to_aa_tuple
    )

    df_out["descriptor_matrix"] = df_out["pocket"].apply(
        pocket_to_descriptor_matrix
    )

    return df_out


def add_pocket_representations_from_position_columns(
    df,
    pos_cols={
        99: "aa99",
        134: "aa134",
        304: "aa304",
        330: "aa330",
    },
):
    """
    Adds ANOVA-GP compatible pocket representations from one amino-acid column per position.
    """
    df_out = df.copy()
    pockets = []

    for _, row in df_out.iterrows():
        pocket = {}

        for position, col in pos_cols.items():
            if col in df_out.columns and not pd.isna(row[col]):
                pocket[position] = str(row[col]).upper()
            else:
                pocket[position] = WT_POCKET[position]

        pockets.append(pocket)

    df_out["pocket"] = pockets

    df_out["aa_tuple"] = df_out["pocket"].apply(
        pocket_to_aa_tuple
    )

    df_out["descriptor_matrix"] = df_out["pocket"].apply(
        pocket_to_descriptor_matrix
    )

    return df_out


# Anwendung auf deinen Datensatz:
df_gp = add_pocket_representations_from_mutation_column(
    df,
    mutation_col="mutations",
)

position_arrays = pockets_to_position_descriptor_arrays(
    df_gp["pocket"].tolist()
)


# Sanity checks:
print("df_gp shape:", df_gp.shape)

print(
    df_gp[
        ["variant_id", "mutations", "pocket", "aa_tuple"]
    ].head()
)

for position, array in position_arrays.items():
    print(position, array.shape)


example_variant = {99: "R", 134: "W", 304: "M", 330: "A"}

example_aa_tuple = pocket_to_aa_tuple(example_variant)
example_descriptor_matrix = pocket_to_descriptor_matrix(example_variant)

print("Example aa_tuple:", example_aa_tuple)
print("Example descriptor matrix shape:", example_descriptor_matrix.shape)
print(example_descriptor_matrix)

df_gp shape: (36, 18)
  variant_id mutations                                   pocket      aa_tuple
0         WT       NaN  {99: 'D', 134: 'F', 304: 'D', 330: 'R'}  (D, F, D, R)
1       D99G      D99G  {99: 'G', 134: 'F', 304: 'D', 330: 'R'}  (G, F, D, R)
2       D99V      D99V  {99: 'V', 134: 'F', 304: 'D', 330: 'R'}  (V, F, D, R)
3       D99R      D99R  {99: 'R', 134: 'F', 304: 'D', 330: 'R'}  (R, F, D, R)
4      F134W     F134W  {99: 'D', 134: 'W', 304: 'D', 330: 'R'}  (D, W, D, R)
99 (36, 8)
134 (36, 8)
304 (36, 8)
330 (36, 8)
Example aa_tuple: ('R', 'W', 'M', 'A')
Example descriptor matrix shape: (4, 8)
[[ 2.22225028 -1.37737267  0.79660011  0.82939936 -0.5         1.
   1.1055416   1.26381432]
 [-0.01116709 -0.14082863  2.14492139 -1.11539914  2.          1.
  -0.90453403 -1.50618967]
 [-0.01116709  0.82092785  0.53635427 -1.00099923 -0.5        -1.
  -0.90453403 -1.62985056]
 [-0.01116709  0.7865794  -1.30519483 -0.08579993 -0.5        -1.
  -0.90453403 -0.86315303]]


### Build position-specific RBF kernels

This cell defines the radial basis function kernel used to measure similarity between amino-acid descriptor vectors. It then wraps this kernel into a helper that constructs one separate kernel matrix for each modeled pocket position. These position-specific kernels are the basic building blocks for the additive and epistatic ANOVA-GP components.


In [6]:
def rbf_kernel_from_descriptors(X1, X2=None, lengthscale=1.0):
    """
    Computes an RBF kernel between amino-acid descriptor vectors.

    X1 shape: (n_variants_1, n_properties)
    X2 shape: (n_variants_2, n_properties)

    Returns:
    K shape: (n_variants_1, n_variants_2)
    """
    if X2 is None:
        X2 = X1

    squared_distances = (
        (X1[:, None, :] - X2[None, :, :]) ** 2
    ).sum(axis=2)

    K = np.exp(
        -squared_distances / (2 * lengthscale**2)
    )

    return K


def build_position_kernels(position_arrays, lengthscales=None):
    """
    Builds one RBF kernel per mutation position.

    position_arrays:
    dictionary position -> descriptor matrix

    lengthscales:
    dictionary position -> kernel lengthscale
    """
    if lengthscales is None:
        lengthscales = {
            position: 1.0
            for position in position_arrays
        }

    position_kernels = {}

    for position, X_pos in position_arrays.items():
        position_kernels[position] = rbf_kernel_from_descriptors(
            X_pos,
            lengthscale=lengthscales[position],
        )

    return position_kernels

### Compute initial position-specific kernels

This cell applies the position-specific RBF kernel construction to the descriptor arrays generated from the variant pockets. A shared initial lengthscale of 1.0 is used for all four positions. The printed shape and value range check that each position produces a valid square kernel matrix over all measured variants.


In [7]:
position_kernels = build_position_kernels(
    position_arrays,
    lengthscales={
        99: 1.0,
        134: 1.0,
        304: 1.0,
        330: 1.0,
    },
)

for position, K in position_kernels.items():
    print(position, K.shape, K.min(), K.max())

99 (36, 36) 5.44191952770634e-07 1.0
134 (36, 36) 0.03846958506715717 1.0
304 (36, 36) 3.8978655099074254e-10 1.0
330 (36, 36) 1.0728587144032161e-06 1.0


### Check kernel self-similarity

This cell verifies that the diagonal entries of each position-specific kernel are equal to one. This is expected for an RBF kernel because every variant is maximally similar to itself at a given position.


In [8]:
for position, K in position_kernels.items():
    print(position, np.allclose(np.diag(K), 1.0))

99 True
134 True
304 True
330 True


### Construct additive, epistatic, and total ANOVA kernels

This cell defines the kernel composition used by the ANOVA-GP. The main-effect kernel averages the four position-specific kernels, while the epistasis kernel averages all pairwise products between position-specific kernels. The total kernel combines both components with separate amplitudes for additive and pairwise-interaction effects.


In [9]:
import itertools
import numpy as np


def build_main_kernel(position_kernels):
    """
    Additive main-effect kernel:
    k_main = mean over position-specific kernels
    """
    kernels = list(position_kernels.values())

    k_main = np.mean(
        np.stack(kernels, axis=0),
        axis=0,
    )

    return k_main


def build_epistasis_kernel(position_kernels):
    """
    Pairwise ANOVA epistasis kernel:
    k_epi = mean over pairwise products of position-specific kernels
    """
    positions = list(position_kernels.keys())
    pairwise_products = []

    for pos_a, pos_b in itertools.combinations(positions, 2):
        pairwise_products.append(
            position_kernels[pos_a] * position_kernels[pos_b]
        )

    k_epi = np.mean(
        np.stack(pairwise_products, axis=0),
        axis=0,
    )

    return k_epi


def build_total_anova_kernel(
    position_kernels,
    sigma_main=1.0,
    sigma_epi=0.5,
):
    """
    Total ANOVA-GP kernel:
    k_total = sigma_main^2 * k_main + sigma_epi^2 * k_epi
    """
    k_main = build_main_kernel(position_kernels)
    k_epi = build_epistasis_kernel(position_kernels)

    k_total = (
        sigma_main**2 * k_main
        + sigma_epi**2 * k_epi
    )

    return k_total, k_main, k_epi


K_total, K_main, K_epi = build_total_anova_kernel(
    position_kernels,
    sigma_main=1.0,
    sigma_epi=0.5,
)


print("K_main:", K_main.shape, K_main.min(), K_main.max())
print("K_epi:", K_epi.shape, K_epi.min(), K_epi.max())
print("K_total:", K_total.shape, K_total.min(), K_total.max())

print("K_main symmetric:", np.allclose(K_main, K_main.T))
print("K_epi symmetric:", np.allclose(K_epi, K_epi.T))
print("K_total symmetric:", np.allclose(K_total, K_total.T))

print("K_main diagonal:", np.unique(np.round(np.diag(K_main), 6)))
print("K_epi diagonal:", np.unique(np.round(np.diag(K_epi), 6)))
print("K_total diagonal:", np.unique(np.round(np.diag(K_total), 6)))

K_main: (36, 36) 0.009617926550274677 1.0
K_epi: (36, 36) 1.3600090262039097e-08 1.0
K_total: (36, 36) 0.009617929950297243 1.25
K_main symmetric: True
K_epi symmetric: True
K_total symmetric: True
K_main diagonal: [1.]
K_epi diagonal: [1.]
K_total diagonal: [1.25]


### Fit an exact Gaussian process from a precomputed kernel

This cell defines helper functions for target standardization, exact GP fitting with a precomputed covariance matrix, and posterior prediction at the training points. The GP covariance includes the ANOVA kernel, the squared standard error of the activity mean, an additional noise term, and a small jitter term for numerical stability.


In [10]:
import numpy as np
import pandas as pd


def standardize_target(y):
    y = np.asarray(y, dtype=float)

    y_mean = y.mean()
    y_std = y.std(ddof=1)

    y_scaled = (y - y_mean) / y_std

    return y_scaled, y_mean, y_std


def fit_gp_from_precomputed_kernel(
    K_total,
    y,
    sem=None,
    sigma_noise=0.1,
    jitter=1e-8,
):
    """
    Fits an exact GP using a precomputed kernel.

    K_y = K_total + diag(sem_scaled^2) + sigma_noise^2 I

    y is standardized internally.
    sem is transformed to the same standardized scale.
    """
    y_scaled, y_mean, y_std = standardize_target(y)

    n = len(y_scaled)

    if sem is None:
        sem_scaled = np.zeros(n)
    else:
        sem = np.asarray(sem, dtype=float)
        sem_scaled = sem / y_std

    K_y = (
        K_total
        + np.diag(sem_scaled**2)
        + sigma_noise**2 * np.eye(n)
        + jitter * np.eye(n)
    )

    L = np.linalg.cholesky(K_y)

    alpha = np.linalg.solve(
        L.T,
        np.linalg.solve(L, y_scaled)
    )

    gp_fit = {
        "K_total": K_total,
        "K_y": K_y,
        "L": L,
        "alpha": alpha,
        "y_mean": y_mean,
        "y_std": y_std,
        "y_scaled": y_scaled,
        "sem_scaled": sem_scaled,
        "sigma_noise": sigma_noise,
        "jitter": jitter,
    }

    return gp_fit


def gp_predict_training_points(gp_fit):
    """
    Posterior mean and variance for the training points.
    This is mainly a sanity check, not valid model evaluation.
    """
    K_train = gp_fit["K_total"]
    L = gp_fit["L"]
    alpha = gp_fit["alpha"]

    mean_scaled = K_train @ alpha

    v = np.linalg.solve(L, K_train.T)
    cov_scaled = K_train - v.T @ v

    mean = (
        gp_fit["y_mean"]
        + gp_fit["y_std"] * mean_scaled
    )

    std = gp_fit["y_std"] * np.sqrt(
        np.maximum(np.diag(cov_scaled), 0)
    )

    return mean, std


y = df_gp["activity_pa6"].to_numpy(dtype=float)
sem = df_gp["activity_sem"].to_numpy(dtype=float)

gp_fit = fit_gp_from_precomputed_kernel(
    K_total=K_total,
    y=y,
    sem=sem,
    sigma_noise=0.1,
)

train_mean, train_std = gp_predict_training_points(gp_fit)

training_fit_df = pd.DataFrame({
    "variant_id": df_gp["variant_id"],
    "observed": y,
    "gp_train_mean": train_mean,
    "gp_train_std": train_std,
    "abs_error": np.abs(y - train_mean),
})

display(training_fit_df.head(10))

print("Training MAE:", training_fit_df["abs_error"].mean())

,variant_id,observed,gp_train_mean,gp_train_std,abs_error
0,WT,77.666667,99.956146,6.958810,22.289479
1,D99G,144.000000,143.564005,14.603345,0.435995
2,D99V,147.666667,143.744760,10.817684,3.921907
3,D99R,194.000000,190.770261,12.123736,3.229739
4,F134W,217.000000,218.818972,9.821015,1.818972
5,F301L,118.500000,99.956146,6.958810,18.543854
6,D304M,233.500000,232.781924,9.643648,0.718076
7,D304E,171.250000,159.954443,10.337359,11.295557
8,D304Q,157.000000,156.458826,12.148347,0.541174
9,D304V,131.500000,108.218474,14.453234,23.281526


Training MAE: 8.807170878662726


### Exclude variants outside the four-position modeling space

This cell extracts all mutated positions from each mutation string and flags variants whose substitutions lie outside the modeled target positions. The core dataset is then restricted to variants whose mutations are fully contained within positions 99, 134, 304, and 330. This avoids treating external mutations, such as F301L, as if they were identical to the wild type in the four-position representation.


In [11]:
import re


TARGET_POSITIONS = set(POSITIONS)


def mutation_positions(mutation_string):
    """
    Extracts all mutated positions from a mutation string.
    Returns an empty set for WT.
    """
    if pd.isna(mutation_string) or str(mutation_string).strip().lower() in [
        "",
        "wt",
        "wildtype",
        "wild type",
    ]:
        return set()

    matches = re.findall(
        r"([A-Z])(\d+)([A-Z])",
        str(mutation_string).upper(),
    )

    return {int(pos) for _, pos, _ in matches}


df_gp["mutation_positions"] = df_gp["mutations"].apply(
    mutation_positions
)

df_gp["only_target_positions"] = df_gp["mutation_positions"].apply(
    lambda positions: positions.issubset(TARGET_POSITIONS)
)

df_gp["has_external_mutation"] = ~df_gp["only_target_positions"]

print(
    df_gp[
        [
            "variant_id",
            "mutations",
            "mutation_positions",
            "only_target_positions",
            "has_external_mutation",
        ]
    ]
)

df_core = df_gp[df_gp["only_target_positions"]].copy()

print("Full dataset:", df_gp.shape)
print("Core dataset:", df_core.shape)
print("Excluded variants:")
display(
    df_gp.loc[
        df_gp["has_external_mutation"],
        ["variant_id", "mutations", "mutation_positions"]
    ]
)

                variant_id               mutations   mutation_positions  \
0                       WT                     NaN                   {}   
1                     D99G                    D99G                 {99}   
2                     D99V                    D99V                 {99}   
3                     D99R                    D99R                 {99}   
4                    F134W                   F134W                {134}   
5                    F301L                   F301L                {301}   
6                    D304M                   D304M                {304}   
7                    D304E                   D304E                {304}   
8                    D304Q                   D304Q                {304}   
9                    D304V                   D304V                {304}   
10                   D304W                   D304W                {304}   
11                   D304R                   D304R                {304}   
12                   D304

,variant_id,mutations,mutation_positions
5,F301L,F301L,{301}


### Fit the ANOVA-GP on the core four-position dataset

This cell recomputes descriptor arrays and ANOVA kernels for the curated core dataset, fits the GP using fixed initial hyperparameters, and predicts the training points as a sanity check. The resulting training fit is not used as a generalization estimate, but it verifies that the kernel and GP implementation can represent the observed activity values.


In [12]:
core_position_arrays = pockets_to_position_descriptor_arrays(
    df_core["pocket"].tolist()
)

core_position_kernels = build_position_kernels(
    core_position_arrays,
    lengthscales={
        99: 1.0,
        134: 1.0,
        304: 1.0,
        330: 1.0,
    },
)

K_core_total, K_core_main, K_core_epi = build_total_anova_kernel(
    core_position_kernels,
    sigma_main=1.0,
    sigma_epi=0.5,
)

y_core = df_core["activity_pa6"].to_numpy(dtype=float)
sem_core = df_core["activity_sem"].to_numpy(dtype=float)

gp_core_fit = fit_gp_from_precomputed_kernel(
    K_total=K_core_total,
    y=y_core,
    sem=sem_core,
    sigma_noise=0.1,
)

core_train_mean, core_train_std = gp_predict_training_points(
    gp_core_fit
)

core_training_fit_df = pd.DataFrame({
    "variant_id": df_core["variant_id"].values,
    "observed": y_core,
    "gp_train_mean": core_train_mean,
    "gp_train_std": core_train_std,
    "abs_error": np.abs(y_core - core_train_mean),
})

display(core_training_fit_df.head(10))

print("Core training MAE:", core_training_fit_df["abs_error"].mean())

,variant_id,observed,gp_train_mean,gp_train_std,abs_error
0,WT,77.666667,87.966082,8.887764,10.299415
1,D99G,144.000000,143.036459,14.573383,0.963541
2,D99V,147.666667,143.450527,10.777187,4.216139
3,D99R,194.000000,187.639529,12.177333,6.360471
4,F134W,217.000000,215.221107,9.927917,1.778893
5,D304M,233.500000,231.259307,9.631993,2.240693
6,D304E,171.250000,158.341296,10.326850,12.908704
7,D304Q,157.000000,156.378973,12.110563,0.621027
8,D304V,131.500000,107.795778,14.422567,23.704222
9,D304W,130.000000,130.668105,11.476046,0.668105


Core training MAE: 8.495579029415023


### Define cross-kernel prediction and LOOCV utilities

This cell defines helper functions for building cross-kernels between held-out test variants and training variants, predicting from a fitted GP, and running leave-one-variant-out cross-validation with fixed hyperparameters. These functions enable retrospective evaluation of the ANOVA-GP without using the held-out variant during model fitting.


In [13]:
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr


def build_position_arrays_for_pockets(pockets, positions=POSITIONS):
    return pockets_to_position_descriptor_arrays(
        pockets,
        positions=positions,
    )


def build_position_kernels_between(
    train_pockets,
    test_pockets,
    lengthscales=None,
    positions=POSITIONS,
):
    """
    Builds position-specific cross-kernels between test and train pockets.

    Returns:
    position -> K_test_train with shape (n_test, n_train)
    """
    if lengthscales is None:
        lengthscales = {
            position: 1.0
            for position in positions
        }

    train_arrays = pockets_to_position_descriptor_arrays(
        train_pockets,
        positions=positions,
    )

    test_arrays = pockets_to_position_descriptor_arrays(
        test_pockets,
        positions=positions,
    )

    cross_kernels = {}

    for position in positions:
        cross_kernels[position] = rbf_kernel_from_descriptors(
            test_arrays[position],
            X2=train_arrays[position],
            lengthscale=lengthscales[position],
        )

    return cross_kernels


def build_total_anova_cross_kernel(
    train_pockets,
    test_pockets,
    lengthscales=None,
    sigma_main=1.0,
    sigma_epi=0.5,
    positions=POSITIONS,
):
    """
    Builds K_test_train for the ANOVA kernel.
    """
    cross_position_kernels = build_position_kernels_between(
        train_pockets=train_pockets,
        test_pockets=test_pockets,
        lengthscales=lengthscales,
        positions=positions,
    )

    k_main_cross = build_main_kernel(
        cross_position_kernels
    )

    k_epi_cross = build_epistasis_kernel(
        cross_position_kernels
    )

    k_total_cross = (
        sigma_main**2 * k_main_cross
        + sigma_epi**2 * k_epi_cross
    )

    return k_total_cross


def gp_predict_from_fit(
    gp_fit,
    K_test_train,
    K_test_test_diag=None,
):
    """
    Predicts GP posterior mean and std for test variants.

    K_test_train shape:
    (n_test, n_train)
    """
    alpha = gp_fit["alpha"]
    L = gp_fit["L"]

    mean_scaled = K_test_train @ alpha

    mean = (
        gp_fit["y_mean"]
        + gp_fit["y_std"] * mean_scaled
    )

    if K_test_test_diag is None:
        K_test_test_diag = np.ones(K_test_train.shape[0]) * 1.25

    v = np.linalg.solve(L, K_test_train.T)

    var_scaled = K_test_test_diag - np.sum(v**2, axis=0)
    var_scaled = np.maximum(var_scaled, 0)

    std = gp_fit["y_std"] * np.sqrt(var_scaled)

    return mean, std


def loocv_anova_gp_fixed_hyperparameters(
    df_core,
    lengthscales=None,
    sigma_main=1.0,
    sigma_epi=0.5,
    sigma_noise=0.1,
):
    if lengthscales is None:
        lengthscales = {
            99: 1.0,
            134: 1.0,
            304: 1.0,
            330: 1.0,
        }

    pockets = df_core["pocket"].tolist()
    y = df_core["activity_pa6"].to_numpy(dtype=float)
    sem = df_core["activity_sem"].to_numpy(dtype=float)

    predictions = []
    uncertainties = []

    for test_idx in range(len(df_core)):
        train_idx = [
            i for i in range(len(df_core))
            if i != test_idx
        ]

        train_pockets = [pockets[i] for i in train_idx]
        test_pocket = [pockets[test_idx]]

        y_train = y[train_idx]
        sem_train = sem[train_idx]

        train_position_arrays = pockets_to_position_descriptor_arrays(
            train_pockets
        )

        train_position_kernels = build_position_kernels(
            train_position_arrays,
            lengthscales=lengthscales,
        )

        K_train_total, _, _ = build_total_anova_kernel(
            train_position_kernels,
            sigma_main=sigma_main,
            sigma_epi=sigma_epi,
        )

        gp_fit = fit_gp_from_precomputed_kernel(
            K_total=K_train_total,
            y=y_train,
            sem=sem_train,
            sigma_noise=sigma_noise,
        )

        K_test_train = build_total_anova_cross_kernel(
            train_pockets=train_pockets,
            test_pockets=test_pocket,
            lengthscales=lengthscales,
            sigma_main=sigma_main,
            sigma_epi=sigma_epi,
        )

        K_test_test_diag = np.array([
            sigma_main**2 + sigma_epi**2
        ])

        pred_mean, pred_std = gp_predict_from_fit(
            gp_fit,
            K_test_train=K_test_train,
            K_test_test_diag=K_test_test_diag,
        )

        predictions.append(pred_mean[0])
        uncertainties.append(pred_std[0])

    predictions = np.array(predictions)
    uncertainties = np.array(uncertainties)

    result_df = pd.DataFrame({
        "variant_id": df_core["variant_id"].values,
        "observed": y,
        "predicted": predictions,
        "predicted_std": uncertainties,
        "abs_error": np.abs(y - predictions),
    })

    metrics = {
        "mae": mean_absolute_error(y, predictions),
        "r2": r2_score(y, predictions),
        "pearson": pearsonr(y, predictions).statistic,
        "spearman": spearmanr(y, predictions).statistic,
    }

    return result_df, metrics

### Run an initial fixed-hyperparameter LOOCV evaluation

This cell evaluates the ANOVA-GP with manually chosen initial hyperparameters using leave-one-variant-out cross-validation. The resulting table compares observed and predicted PA6 activity for each held-out variant and reports basic performance metrics.


In [14]:
loocv_result_df, loocv_metrics = loocv_anova_gp_fixed_hyperparameters(
    df_core,
    lengthscales={
        99: 1.0,
        134: 1.0,
        304: 1.0,
        330: 1.0,
    },
    sigma_main=1.0,
    sigma_epi=0.5,
    sigma_noise=0.1,
)

display(loocv_result_df)

loocv_metrics

,variant_id,observed,predicted,predicted_std,abs_error
0,WT,77.666667,112.356175,16.022416,34.689508
1,D99G,144.000000,134.051184,47.355255,9.948816
2,D99V,147.666667,66.365487,47.525366,81.301180
3,D99R,194.000000,172.578400,22.660096,21.421600
4,F134W,217.000000,210.434669,19.285153,6.565331
5,D304M,233.500000,222.609229,21.563085,10.890771
6,D304E,171.250000,101.294827,24.230646,69.955173
7,D304Q,157.000000,138.572003,66.789090,18.427997
8,D304V,131.500000,49.584544,26.787322,81.915456
9,D304W,130.000000,155.094127,69.521113,25.094127


{'mae': 54.437154609383114,
 'r2': 0.571606873229433,
 'pearson': 0.781370486342091,
 'spearman': 0.7539743698019127}

### Define a coarse hyperparameter grid search

This cell defines a grid-search function that evaluates the ANOVA-GP across shared position lengthscales, epistasis amplitudes, and additional noise levels. Each hyperparameter setting is scored by leave-one-variant-out cross-validation, enabling a retrospective sensitivity analysis of the kernel regime.


In [15]:
from itertools import product
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr


def evaluate_anova_gp_hyperparameter_grid(
    df_core,
    lengthscale_values=(0.5, 1.0, 2.0, 4.0),
    sigma_epi_values=(0.0, 0.25, 0.5, 1.0, 2.0),
    sigma_noise_values=(0.05, 0.1, 0.2, 0.5),
    sigma_main=1.0,
):
    rows = []

    for lengthscale, sigma_epi, sigma_noise in product(
        lengthscale_values,
        sigma_epi_values,
        sigma_noise_values,
    ):
        lengthscales = {
            99: lengthscale,
            134: lengthscale,
            304: lengthscale,
            330: lengthscale,
        }

        loocv_result_df, metrics = loocv_anova_gp_fixed_hyperparameters(
            df_core,
            lengthscales=lengthscales,
            sigma_main=sigma_main,
            sigma_epi=sigma_epi,
            sigma_noise=sigma_noise,
        )

        rows.append({
            "lengthscale": lengthscale,
            "sigma_main": sigma_main,
            "sigma_epi": sigma_epi,
            "sigma_noise": sigma_noise,
            "mae": metrics["mae"],
            "r2": metrics["r2"],
            "pearson": metrics["pearson"],
            "spearman": metrics["spearman"],
        })

    grid_results = pd.DataFrame(rows)

    return grid_results.sort_values("mae")


grid_results = evaluate_anova_gp_hyperparameter_grid(
    df_core,
    lengthscale_values=(0.5, 1.0, 2.0, 4.0),
    sigma_epi_values=(0.0, 0.25, 0.5, 1.0, 2.0),
    sigma_noise_values=(0.05, 0.1, 0.2, 0.5),
    sigma_main=1.0,
)

display(grid_results.head(20))

,lengthscale,sigma_main,sigma_epi,sigma_noise,mae,r2,pearson,spearman
19,0.5,1.0,2.0,0.50,46.654664,0.684763,0.828260,0.804958
18,0.5,1.0,2.0,0.20,46.956135,0.671825,0.823924,0.802297
39,1.0,1.0,2.0,0.50,47.125810,0.665136,0.817245,0.814343
35,1.0,1.0,1.0,0.50,47.280331,0.674335,0.821201,0.815323
15,0.5,1.0,1.0,0.50,47.647926,0.680856,0.825148,0.825408
17,0.5,1.0,2.0,0.10,47.850371,0.668439,0.823034,0.795714
55,2.0,1.0,1.0,0.50,48.153381,0.632232,0.795361,0.785769
16,0.5,1.0,2.0,0.05,48.329624,0.665804,0.822210,0.798515
38,1.0,1.0,2.0,0.20,49.225825,0.639128,0.808484,0.798515
59,2.0,1.0,2.0,0.50,49.287881,0.584765,0.771348,0.775825


## nested LOOCV

In [16]:
from itertools import product
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr


def evaluate_hyperparameters_on_subset_loocv(
    df_subset,
    lengthscale,
    sigma_main,
    sigma_epi,
    sigma_noise,
):
    """
    Inner-loop LOOCV evaluation for one hyperparameter setting.
    The function receives only the outer-training subset.
    """
    lengthscales = {
        99: lengthscale,
        134: lengthscale,
        304: lengthscale,
        330: lengthscale,
    }

    inner_result_df, inner_metrics = loocv_anova_gp_fixed_hyperparameters(
        df_subset,
        lengthscales=lengthscales,
        sigma_main=sigma_main,
        sigma_epi=sigma_epi,
        sigma_noise=sigma_noise,
    )

    return inner_metrics["mae"]


def select_hyperparameters_inner_loocv(
    df_train,
    lengthscale_values=(0.2, 0.3, 0.5),
    sigma_main_values=(0.25, 0.5, 1.0),
    sigma_epi_values=(0.0, 1.0, 2.0, 4.0),
    sigma_noise_values=(0.3, 0.5, 0.75, 1.0),
):
    """
    Selects hyperparameters using only the outer-training data.
    The criterion is inner-loop LOOCV MAE.
    """
    rows = []

    for lengthscale, sigma_main, sigma_epi, sigma_noise in product(
        lengthscale_values,
        sigma_main_values,
        sigma_epi_values,
        sigma_noise_values,
    ):
        inner_mae = evaluate_hyperparameters_on_subset_loocv(
            df_subset=df_train,
            lengthscale=lengthscale,
            sigma_main=sigma_main,
            sigma_epi=sigma_epi,
            sigma_noise=sigma_noise,
        )

        rows.append({
            "lengthscale": lengthscale,
            "sigma_main": sigma_main,
            "sigma_epi": sigma_epi,
            "sigma_noise": sigma_noise,
            "inner_mae": inner_mae,
        })

    inner_grid = pd.DataFrame(rows).sort_values(
        "inner_mae",
        ascending=True,
    )

    best_params = inner_grid.iloc[0].to_dict()

    return best_params, inner_grid


def fit_and_predict_one_outer_fold(
    df_train,
    df_test,
    lengthscale,
    sigma_main,
    sigma_epi,
    sigma_noise,
):
    """
    Fits the ANOVA-GP on one outer-training set and predicts one outer-test variant.
    """
    lengthscales = {
        99: lengthscale,
        134: lengthscale,
        304: lengthscale,
        330: lengthscale,
    }

    train_pockets = df_train["pocket"].tolist()
    test_pockets = df_test["pocket"].tolist()

    y_train = df_train["activity_pa6"].to_numpy(dtype=float)
    sem_train = df_train["activity_sem"].to_numpy(dtype=float)

    train_position_arrays = pockets_to_position_descriptor_arrays(
        train_pockets
    )

    train_position_kernels = build_position_kernels(
        train_position_arrays,
        lengthscales=lengthscales,
    )

    K_train_total, _, _ = build_total_anova_kernel(
        train_position_kernels,
        sigma_main=sigma_main,
        sigma_epi=sigma_epi,
    )

    gp_fit = fit_gp_from_precomputed_kernel(
        K_total=K_train_total,
        y=y_train,
        sem=sem_train,
        sigma_noise=sigma_noise,
    )

    K_test_train = build_total_anova_cross_kernel(
        train_pockets=train_pockets,
        test_pockets=test_pockets,
        lengthscales=lengthscales,
        sigma_main=sigma_main,
        sigma_epi=sigma_epi,
    )

    K_test_test_diag = np.full(
        len(df_test),
        sigma_main**2 + sigma_epi**2,
    )

    pred_mean, pred_std = gp_predict_from_fit(
        gp_fit,
        K_test_train=K_test_train,
        K_test_test_diag=K_test_test_diag,
    )

    return pred_mean, pred_std


def nested_loocv_anova_gp(
    df_core,
    lengthscale_values=(0.2, 0.3, 0.5),
    sigma_main_values=(0.25, 0.5, 1.0),
    sigma_epi_values=(0.0, 1.0, 2.0, 4.0),
    sigma_noise_values=(0.3, 0.5, 0.75, 1.0),
):
    """
    Nested leave-one-variant-out cross-validation.

    Outer loop:
    evaluates generalization to a held-out variant.

    Inner loop:
    selects hyperparameters using only the outer-training variants.
    """
    outer_rows = []
    inner_grids = {}

    n = len(df_core)

    for outer_test_idx in range(n):
        df_test = df_core.iloc[[outer_test_idx]].copy()
        df_train = df_core.drop(df_core.index[outer_test_idx]).copy()

        best_params, inner_grid = select_hyperparameters_inner_loocv(
            df_train=df_train,
            lengthscale_values=lengthscale_values,
            sigma_main_values=sigma_main_values,
            sigma_epi_values=sigma_epi_values,
            sigma_noise_values=sigma_noise_values,
        )

        pred_mean, pred_std = fit_and_predict_one_outer_fold(
            df_train=df_train,
            df_test=df_test,
            lengthscale=best_params["lengthscale"],
            sigma_main=best_params["sigma_main"],
            sigma_epi=best_params["sigma_epi"],
            sigma_noise=best_params["sigma_noise"],
        )

        observed = float(df_test["activity_pa6"].iloc[0])
        predicted = float(pred_mean[0])
        predicted_std = float(pred_std[0])

        outer_rows.append({
            "variant_id": df_test["variant_id"].iloc[0],
            "observed": observed,
            "predicted": predicted,
            "predicted_std": predicted_std,
            "abs_error": abs(observed - predicted),
            "selected_lengthscale": best_params["lengthscale"],
            "selected_sigma_main": best_params["sigma_main"],
            "selected_sigma_epi": best_params["sigma_epi"],
            "selected_sigma_noise": best_params["sigma_noise"],
            "inner_mae": best_params["inner_mae"],
        })

        inner_grids[df_test["variant_id"].iloc[0]] = inner_grid

        print(
            outer_test_idx + 1,
            "/",
            n,
            df_test["variant_id"].iloc[0],
            "predicted",
            round(predicted, 2),
            "observed",
            round(observed, 2),
            "abs_error",
            round(abs(observed - predicted), 2),
        )

    nested_result_df = pd.DataFrame(outer_rows)

    y_true = nested_result_df["observed"].to_numpy(dtype=float)
    y_pred = nested_result_df["predicted"].to_numpy(dtype=float)

    nested_metrics = {
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
        "pearson": pearsonr(y_true, y_pred).statistic,
        "spearman": spearmanr(y_true, y_pred).statistic,
    }

    return nested_result_df, nested_metrics, inner_grids

In [17]:
nested_result_df, nested_metrics, nested_inner_grids = nested_loocv_anova_gp(
    df_core,
    lengthscale_values=(0.2, 0.3, 0.5),
    sigma_main_values=(0.25, 0.5, 1.0),
    sigma_epi_values=(0.0, 1.0, 2.0, 4.0),
    sigma_noise_values=(0.3, 0.5, 0.75, 1.0),
)

display(
    nested_result_df.sort_values(
        "abs_error",
        ascending=False,
    )
)

nested_metrics

1 / 35 WT predicted 87.62 observed 77.67 abs_error 9.95
2 / 35 D99G predicted 154.08 observed 144.0 abs_error 10.08
3 / 35 D99V predicted 123.75 observed 147.67 abs_error 23.92
4 / 35 D99R predicted 166.45 observed 194.0 abs_error 27.55
5 / 35 F134W predicted 246.89 observed 217.0 abs_error 29.89
6 / 35 D304M predicted 211.61 observed 233.5 abs_error 21.89
7 / 35 D304E predicted 130.76 observed 171.25 abs_error 40.49
8 / 35 D304Q predicted 160.99 observed 157.0 abs_error 3.99
9 / 35 D304V predicted 108.92 observed 131.5 abs_error 22.58
10 / 35 D304W predicted 164.06 observed 130.0 abs_error 34.06
11 / 35 D304R predicted 143.77 observed 123.0 abs_error 20.77
12 / 35 D304L predicted 231.64 observed 137.5 abs_error 94.14
13 / 35 R330A predicted 244.58 observed 147.0 abs_error 97.58
14 / 35 R330Q predicted 125.12 observed 140.67 abs_error 15.54
15 / 35 F134W_D304M predicted 342.67 observed 409.67 abs_error 67.0
16 / 35 F134W_D304L predicted 247.61 observed 370.33 abs_error 122.72
17 / 35 F

,variant_id,observed,predicted,predicted_std,abs_error,selected_lengthscale,selected_sigma_main,selected_sigma_epi,selected_sigma_noise,inner_mae
23,D99R_F134W_R330A,406.000000,263.929730,209.184504,142.070270,0.2,0.25,4.0,0.30,37.222711
15,F134W_D304L,370.333333,247.608585,238.014697,122.724748,0.2,0.25,4.0,0.50,38.941983
29,D99R_F134W_D304M_R330A,296.000000,407.105230,157.602788,111.105230,0.2,0.25,4.0,0.30,32.487366
12,R330A,147.000000,244.583052,245.613241,97.583052,0.2,1.00,4.0,0.30,43.050112
11,D304L,137.500000,231.641760,240.403030,94.141760,0.2,0.25,4.0,0.50,41.761600
20,F134W_D304V,202.333333,291.743589,230.174930,89.410256,0.5,0.25,4.0,0.50,43.246244
31,D99V_F134W_D304M_R330A,391.000000,307.696096,221.735949,83.303904,0.2,0.25,4.0,0.50,42.717075
16,F134W_D304I,341.333333,269.976866,313.778167,71.356467,0.2,0.25,4.0,0.50,42.653778
26,D99V_F134W_D304M,201.000000,268.682508,205.873732,67.682508,0.2,0.25,4.0,0.50,44.183168
14,F134W_D304M,409.666667,342.670015,44.828431,66.996652,0.2,0.25,1.0,0.50,46.888435


{'mae': 49.67249392159977,
 'r2': 0.6588171005694146,
 'pearson': 0.8120918330103921,
 'spearman': 0.8070593198585586}

In [18]:
selected_param_summary = (
    nested_result_df[
        [
            "selected_lengthscale",
            "selected_sigma_main",
            "selected_sigma_epi",
            "selected_sigma_noise",
        ]
    ]
    .value_counts()
    .reset_index(name="count")
)

display(selected_param_summary)

,selected_lengthscale,selected_sigma_main,selected_sigma_epi,selected_sigma_noise,count
0,0.2,0.25,4.0,0.50,20
1,0.2,0.25,2.0,0.75,5
2,0.2,0.25,4.0,0.30,2
3,0.2,0.25,2.0,0.30,2
4,0.2,0.25,1.0,0.50,1
5,0.2,0.25,2.0,1.00,1
6,0.2,0.25,4.0,1.00,1
7,0.2,0.50,2.0,0.75,1
8,0.2,1.00,4.0,0.30,1
9,0.5,0.25,4.0,0.50,1


### Inspect the best coarse-grid hyperparameter setting

This cell selects the best-performing hyperparameter combination from the coarse grid according to the sorted grid result table. It provides an initial estimate of which lengthscale, epistasis amplitude, and noise level improve retrospective prediction performance.


In [19]:
best = grid_results.iloc[0]

best

lengthscale     0.500000
sigma_main      1.000000
sigma_epi       2.000000
sigma_noise     0.500000
mae            46.654664
r2              0.684763
pearson         0.828260
spearman        0.804958
Name: 19, dtype: float64

### Run a refined hyperparameter search around the promising regime

This cell evaluates a refined grid focused on shorter lengthscales, stronger epistasis amplitudes, and moderate-to-high noise levels. This follows from the coarse grid results and tests whether performance improves in a more local and interaction-dominated kernel regime.


In [20]:
grid_results_refined = evaluate_anova_gp_hyperparameter_grid(
    df_core,
    lengthscale_values=(0.2, 0.3, 0.5, 0.75, 1.0),
    sigma_epi_values=(1.0, 2.0, 3.0, 4.0),
    sigma_noise_values=(0.3, 0.5, 0.75, 1.0),
    sigma_main=1.0,
)

display(grid_results_refined.head(20))

,lengthscale,sigma_main,sigma_epi,sigma_noise,mae,r2,pearson,spearman
13,0.2,1.0,4.0,0.50,44.365287,0.697921,0.836263,0.821206
29,0.3,1.0,4.0,0.50,44.577697,0.695802,0.835089,0.815043
8,0.2,1.0,3.0,0.30,44.663393,0.695494,0.835371,0.824427
14,0.2,1.0,4.0,0.75,44.775192,0.699961,0.836949,0.822046
12,0.2,1.0,4.0,0.30,44.806384,0.696800,0.835959,0.825408
24,0.3,1.0,3.0,0.30,44.876495,0.693324,0.834195,0.818265
9,0.2,1.0,3.0,0.50,44.886622,0.697565,0.835913,0.819245
30,0.3,1.0,4.0,0.75,44.938929,0.698008,0.835831,0.817564
15,0.2,1.0,4.0,1.00,44.962094,0.703037,0.838496,0.819805
28,0.3,1.0,4.0,0.30,45.023962,0.694621,0.834776,0.822887


### Separate additive and epistatic refined-grid results

This cell attempts to split the refined hyperparameter results into additive and epistatic models. Because the refined grid contains only positive epistasis amplitudes, the additive subset may be empty; a dedicated ablation grid including sigma_epi = 0 is therefore computed in the next step.


In [21]:
additive_results = grid_results_refined[
    grid_results_refined["sigma_epi"] == 0
].sort_values("mae")

epistatic_results = grid_results_refined[
    grid_results_refined["sigma_epi"] > 0
].sort_values("mae")

display(additive_results.head())
display(epistatic_results.head())

,lengthscale,sigma_main,sigma_epi,sigma_noise,mae,r2,pearson,spearman


,lengthscale,sigma_main,sigma_epi,sigma_noise,mae,r2,pearson,spearman
13,0.2,1.0,4.0,0.50,44.365287,0.697921,0.836263,0.821206
29,0.3,1.0,4.0,0.50,44.577697,0.695802,0.835089,0.815043
8,0.2,1.0,3.0,0.30,44.663393,0.695494,0.835371,0.824427
14,0.2,1.0,4.0,0.75,44.775192,0.699961,0.836949,0.822046
12,0.2,1.0,4.0,0.30,44.806384,0.696800,0.835959,0.825408


### Run an additive-versus-epistatic ablation grid

This cell evaluates a grid that explicitly includes sigma_epi = 0 as the additive baseline and positive sigma_epi values as epistatic models. This provides a direct retrospective comparison between a purely additive physicochemical GP and an ANOVA-GP with pairwise interaction terms.


In [22]:
grid_results_ablation = evaluate_anova_gp_hyperparameter_grid(
    df_core,
    lengthscale_values=(0.2, 0.3, 0.5, 0.75, 1.0),
    sigma_epi_values=(0.0, 1.0, 2.0, 3.0, 4.0),
    sigma_noise_values=(0.3, 0.5, 0.75, 1.0),
    sigma_main=1.0,
)

display(grid_results_ablation.sort_values("mae").head(20))

,lengthscale,sigma_main,sigma_epi,sigma_noise,mae,r2,pearson,spearman
17,0.2,1.0,4.0,0.50,44.365287,0.697921,0.836263,0.821206
37,0.3,1.0,4.0,0.50,44.577697,0.695802,0.835089,0.815043
12,0.2,1.0,3.0,0.30,44.663393,0.695494,0.835371,0.824427
18,0.2,1.0,4.0,0.75,44.775192,0.699961,0.836949,0.822046
16,0.2,1.0,4.0,0.30,44.806384,0.696800,0.835959,0.825408
32,0.3,1.0,3.0,0.30,44.876495,0.693324,0.834195,0.818265
13,0.2,1.0,3.0,0.50,44.886622,0.697565,0.835913,0.819245
38,0.3,1.0,4.0,0.75,44.938929,0.698008,0.835831,0.817564
19,0.2,1.0,4.0,1.00,44.962094,0.703037,0.838496,0.819805
36,0.3,1.0,4.0,0.30,45.023962,0.694621,0.834776,0.822887


### Extend the grid over main-effect and epistasis amplitudes

This cell defines and runs an extended hyperparameter grid that varies both the main-effect amplitude and the epistasis amplitude. This is important because model behavior depends on the relative weighting of additive and pairwise-interaction components, not only on the absolute epistasis amplitude.


In [23]:
def evaluate_anova_gp_hyperparameter_grid_extended(
    df_core,
    lengthscale_values=(0.2, 0.3, 0.5),
    sigma_main_values=(0.25, 0.5, 1.0, 2.0),
    sigma_epi_values=(0.0, 0.5, 1.0, 2.0, 4.0),
    sigma_noise_values=(0.3, 0.5, 0.75, 1.0),
):
    rows = []

    for lengthscale in lengthscale_values:
        for sigma_main in sigma_main_values:
            for sigma_epi in sigma_epi_values:
                for sigma_noise in sigma_noise_values:

                    lengthscales = {
                        99: lengthscale,
                        134: lengthscale,
                        304: lengthscale,
                        330: lengthscale,
                    }

                    loocv_result_df, metrics = loocv_anova_gp_fixed_hyperparameters(
                        df_core,
                        lengthscales=lengthscales,
                        sigma_main=sigma_main,
                        sigma_epi=sigma_epi,
                        sigma_noise=sigma_noise,
                    )

                    rows.append({
                        "lengthscale": lengthscale,
                        "sigma_main": sigma_main,
                        "sigma_epi": sigma_epi,
                        "sigma_noise": sigma_noise,
                        "mae": metrics["mae"],
                        "r2": metrics["r2"],
                        "pearson": metrics["pearson"],
                        "spearman": metrics["spearman"],
                    })

    return pd.DataFrame(rows).sort_values("mae")


grid_results_extended = evaluate_anova_gp_hyperparameter_grid_extended(
    df_core,
    lengthscale_values=(0.2, 0.3, 0.5),
    sigma_main_values=(0.25, 0.5, 1.0, 2.0),
    sigma_epi_values=(0.0, 0.5, 1.0, 2.0, 4.0),
    sigma_noise_values=(0.3, 0.5, 0.75, 1.0),
)

display(grid_results_extended.head(30))

,lengthscale,sigma_main,sigma_epi,sigma_noise,mae,r2,pearson,spearman
17,0.2,0.25,4.0,0.50,44.032085,0.699552,0.836909,0.821206
37,0.2,0.50,4.0,0.50,44.100552,0.699263,0.836797,0.821206
97,0.3,0.25,4.0,0.50,44.247864,0.697463,0.835733,0.821486
12,0.2,0.25,2.0,0.30,44.307371,0.699368,0.836682,0.821486
117,0.3,0.50,4.0,0.50,44.315611,0.697167,0.835621,0.821486
57,0.2,1.00,4.0,0.50,44.365287,0.697921,0.836263,0.821206
16,0.2,0.25,4.0,0.30,44.480186,0.698707,0.836660,0.822326
18,0.2,0.25,4.0,0.75,44.481261,0.701284,0.837572,0.821766
92,0.3,0.25,2.0,0.30,44.487207,0.697354,0.835537,0.818125
38,0.2,0.50,4.0,0.75,44.541164,0.701057,0.837465,0.821766


### Summarize additive versus epistatic model performance

This cell groups the extended grid results into additive and epistatic model classes and reports the best retrospective performance within each class. The summary is used to assess whether pairwise interaction terms improve activity prediction relative to a purely additive kernel.


In [24]:
ablation_summary = (
    grid_results_extended
    .assign(model_type=lambda d: np.where(d["sigma_epi"] == 0, "additive", "epistatic"))
    .groupby("model_type")
    .agg(
        best_mae=("mae", "min"),
        best_r2=("r2", "max"),
        best_spearman=("spearman", "max"),
    )
    .reset_index()
)

display(ablation_summary)

,model_type,best_mae,best_r2,best_spearman
0,additive,58.556642,0.542029,0.792492
1,epistatic,44.032085,0.706995,0.839415


### Display the best additive and epistatic hyperparameter settings

This cell lists the top-performing additive and epistatic models from the extended grid. The comparison makes the difference between the best additive baseline and the best interaction-aware ANOVA-GP explicit.


In [25]:
best_additive = (
    grid_results_extended[
        grid_results_extended["sigma_epi"] == 0
    ]
    .sort_values("mae")
    .head(10)
)

best_epistatic = (
    grid_results_extended[
        grid_results_extended["sigma_epi"] > 0
    ]
    .sort_values("mae")
    .head(10)
)

display(best_additive)
display(best_epistatic)

,lengthscale,sigma_main,sigma_epi,sigma_noise,mae,r2,pearson,spearman
21,0.2,0.50,0.0,0.50,58.556642,0.537430,0.738520,0.777645
101,0.3,0.50,0.0,0.50,58.581671,0.537291,0.738408,0.777645
43,0.2,1.00,0.0,1.00,58.605208,0.537419,0.738395,0.777645
0,0.2,0.25,0.0,0.30,58.618298,0.522424,0.737135,0.782268
123,0.3,1.00,0.0,1.00,58.631007,0.537274,0.738277,0.777645
80,0.3,0.25,0.0,0.30,58.637877,0.522329,0.737040,0.782268
181,0.5,0.50,0.0,0.50,58.724854,0.536892,0.738046,0.771623
160,0.5,0.25,0.0,0.30,58.753285,0.522163,0.736771,0.778626
203,0.5,1.00,0.0,1.00,58.778408,0.536852,0.737894,0.773163
42,0.2,1.00,0.0,0.75,59.068337,0.542029,0.736474,0.780027


,lengthscale,sigma_main,sigma_epi,sigma_noise,mae,r2,pearson,spearman
17,0.2,0.25,4.0,0.50,44.032085,0.699552,0.836909,0.821206
37,0.2,0.50,4.0,0.50,44.100552,0.699263,0.836797,0.821206
97,0.3,0.25,4.0,0.50,44.247864,0.697463,0.835733,0.821486
12,0.2,0.25,2.0,0.30,44.307371,0.699368,0.836682,0.821486
117,0.3,0.50,4.0,0.50,44.315611,0.697167,0.835621,0.821486
57,0.2,1.00,4.0,0.50,44.365287,0.697921,0.836263,0.821206
16,0.2,0.25,4.0,0.30,44.480186,0.698707,0.836660,0.822326
18,0.2,0.25,4.0,0.75,44.481261,0.701284,0.837572,0.821766
92,0.3,0.25,2.0,0.30,44.487207,0.697354,0.835537,0.818125
38,0.2,0.50,4.0,0.75,44.541164,0.701057,0.837465,0.821766


### Evaluate the selected best ANOVA-GP model in LOOCV

This cell selects the best hyperparameter setting from the extended grid and reruns leave-one-variant-out cross-validation. The resulting table is sorted by absolute prediction error to identify which variants are easiest or hardest for the model to predict.


In [26]:
best = grid_results_extended.iloc[0]

best_lengthscales = {
    99: best["lengthscale"],
    134: best["lengthscale"],
    304: best["lengthscale"],
    330: best["lengthscale"],
}

best_loocv_result_df, best_loocv_metrics = loocv_anova_gp_fixed_hyperparameters(
    df_core,
    lengthscales=best_lengthscales,
    sigma_main=best["sigma_main"],
    sigma_epi=best["sigma_epi"],
    sigma_noise=best["sigma_noise"],
)

display(
    best_loocv_result_df
    .sort_values("abs_error", ascending=False)
)

best_loocv_metrics

,variant_id,observed,predicted,predicted_std,abs_error
23,D99R_F134W_R330A,406.000000,263.073427,220.781248,142.926573
15,F134W_D304L,370.333333,247.608585,238.014697,122.724748
29,D99R_F134W_D304M_R330A,296.000000,408.139284,165.481849,112.139284
11,D304L,137.500000,231.641760,240.403030,94.141760
12,R330A,147.000000,238.009930,253.389720,91.009930
31,D99V_F134W_D304M_R330A,391.000000,307.696096,221.735949,83.303904
16,F134W_D304I,341.333333,269.976866,313.778167,71.356467
26,D99V_F134W_D304M,201.000000,268.682508,205.873732,67.682508
20,F134W_D304V,202.333333,267.101467,243.452007,64.768134
27,F134W_D304M_R330A,520.000000,455.674421,166.877407,64.325579


{'mae': 44.03208536128435,
 'r2': 0.6995522371129856,
 'pearson': 0.8369087161282964,
 'spearman': 0.8212059688182451}

### Analyze prediction error by mutation order

This cell adds mutation order to the LOOCV result table and aggregates prediction errors by the number of substitutions per variant. This analysis checks whether single mutants, double mutants, and higher-order combinations differ in retrospective predictability.


In [27]:
best_loocv_result_df["mutation_order"] = df_core["n_mutations"].values

error_by_order = (
    best_loocv_result_df
    .groupby("mutation_order")
    .agg(
        n=("abs_error", "size"),
        mae=("abs_error", "mean"),
        median_abs_error=("abs_error", "median"),
        mean_predicted_std=("predicted_std", "mean"),
    )
    .reset_index()
)

display(error_by_order)

,mutation_order,n,mae,median_abs_error,mean_predicted_std
0,1,14,28.037962,19.228898,226.636787
1,2,9,50.224327,51.369273,215.752646
2,3,6,59.810001,51.110148,183.595895
3,4,6,56.285429,50.020779,215.573924


### Assess posterior uncertainty calibration

This cell evaluates whether the GP posterior standard deviation behaves like a calibrated error estimate. It reports the fraction of held-out variants whose absolute error falls within one or two predicted standard deviations and computes the Spearman correlation between predicted uncertainty and absolute error.


In [28]:
calibration_df = best_loocv_result_df.copy()

calibration_df["within_1std"] = (
    np.abs(calibration_df["observed"] - calibration_df["predicted"])
    <= calibration_df["predicted_std"]
)

calibration_df["within_2std"] = (
    np.abs(calibration_df["observed"] - calibration_df["predicted"])
    <= 2 * calibration_df["predicted_std"]
)

print("Fraction within 1 std:", calibration_df["within_1std"].mean())
print("Fraction within 2 std:", calibration_df["within_2std"].mean())

print(
    "Spearman uncertainty vs abs error:",
    calibration_df["predicted_std"].corr(
        calibration_df["abs_error"],
        method="spearman",
    )
)

Fraction within 1 std: 1.0
Fraction within 2 std: 1.0
Spearman uncertainty vs abs error: 0.14593837535014006
